# Setup

1. using a standalone env:
```bash
mkdir llm-zoomcamp-hw2 && cd llm-zoomcamp-hw2
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
uv run python ../src/embed/download.py
```
**OR**

2. using the project main env: (assumed here)
```bash
uv add gitsource
uv run python src/embed/download.py # run from the project root
```

# Q1. Embedding a query

Embed the following query:

> How does approximate nearest neighbor search work?

In [ ]:
# TODO: add src parent folder to path
from src.embed.embedder import Embedder
embed = Embedder(path="../models/Xenova/all-MiniLM-L6-v2")

In [ ]:
q1q = "How does approximate nearest neighbor search work?"
q1v = embed.encode(q1q)
print(q1v[0])

The embedder returns a vector of 384 numbers. What's the first value (`v[0]`)?

-   -0.31
-   -0.02
-   0.12
-   0.44

# Loading the data

We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit `8c1834d` so everyone works with the same data.

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

Each document is a dictionary with a `filename` and `content`, and there are 72 pages.

# Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its `content`, and compute the cosine similarity with the query vector from Q1.

In [ ]:
file_name = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc = [d for d in documents if d['filename'] == file_name]
assert len(doc) == 1
content = doc[0]['content']
cv = embed.encode(content)
print(q1v.dot(cv))

What do you get?
-   0.07
-   0.37
-   0.68
-   0.92

# Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

In [ ]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

We embed every chunk's `content` with `encode_batch`, stack the vectors into a matrix `X`, and score the Q1 query against all chunks:

In [ ]:
from tqdm.auto import tqdm
import numpy as np

texts = [chunk['content'] for chunk in chunks]
batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)
scores = X.dot(q1v)

In [ ]:
# print(scores.shape)
# print(scores.argmax())
print(chunks[scores.argmax()]['filename'])

Which file does the highest-scoring chunk belong to (its `filename`)?

-   `02-vector-search/lessons/03-embeddings-dataset.md`
-   `02-vector-search/lessons/06-rag-vector.md`
-   `02-vector-search/lessons/07-sqlitesearch-vector.md`
-   `02-vector-search/lessons/09-onnx-embedder.md`

# Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use `VectorSearch` from minsearch and run a search for the following query:

> What metric do we use to evaluate a search engine?

In [ ]:
from minsearch import VectorSearch
vindex = VectorSearch()
vindex.fit(X, chunks)

In [ ]:
q4q = "What metric do we use to evaluate a search engine?"
q4v = embed.encode(q4q)
results = vindex.search(q4v, num_results=1)
results[0]['filename']

Which file is the `filename` of the first result?

-   `02-vector-search/lessons/04-vector-search.md`
-   `04-evaluation/lessons/05-search-metrics.md`
-   `04-evaluation/lessons/13-llm-as-judge.md`
-   `05-monitoring/lessons/04-metrics.md`

# Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with `Index` from minsearch. Use `content` as a text field.

Run both searches for this query:

> How do I store vectors in PostgreSQL?

In [ ]:
from minsearch import Index
index = Index(
    text_fields=['content'],
    # keyword_fields=['filename']
)
q5q = "How do I store vectors in PostgreSQL?"
q5v = embed.encode(q5q)
index.fit(chunks)
text_result = index.search(q5q, num_results=5)
vec_result = vindex.search(q5v, num_results=5)
# print(text_result)
# print(vec_result)
print(set([x['filename'] for x in vec_result]) - set([x['filename'] for x in text_result]))

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

-   `02-vector-search/lessons/01-intro.md`
-   `02-vector-search/lessons/02-embeddings.md`
-   `02-vector-search/lessons/08-pgvector.md`
-   `03-orchestration/lessons/05-rag.md`

# Q6. Hybrid search

- Both vector and text search have their strengths and weaknesses.
    - Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords.
    - Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

- We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

## Reciprocal Rank Fusion

Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (`rank`, starting at 0) in each list, and we sum the scores across lists with a constant `k = 60`:

```
RRF(d) = sum over lists of  1 / (k + rank(d))
```

"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its `1 / (k + rank)` contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant `k` controls how much the exact rank matters. A larger `k` flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller `k` does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

A document that ranks well in both lists ends up higher than one that's only strong in a single list.

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Now run the query `"How do I give the model access to tools?"` with vector and text search and fuse the results with `rrf`:

In [ ]:
q6q = "How do I give the model access to tools?"
q6v = embed.encode(q6q)
text_results = index.search(q6q, num_results=5)
vector_results = vindex.search(q6v, num_results=5)

results = rrf([vector_results, text_results])
results[0]['filename']

Which file is ranked first after RRF?

-   `01-agentic-rag/lessons/01-intro.md`
-   `01-agentic-rag/lessons/13-function-calling.md`
-   `01-agentic-rag/lessons/14-agentic-loop.md`
-   `01-agentic-rag/lessons/16-other-frameworks.md`

Notice that this file isn't first in either search on its own - it wins because it ranks high in both.



# Selecting the best approach

By now you can search several ways:

-   vector search
-   keyword search
-   hybrid search

Which is the best way? The right choice depends on your data, and the way to decide is to measure. see:
- [evaluation module](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/04-evaluation/lessons/04-search-evaluation.md)
- and [evaluation homework](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/04-evaluation/homework.md).